Paruošiamos bibliotekos, keliai ir pagrindiniai modelio parametrai.


In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd

from pmdarima import auto_arima
from pmdarima.arima import ARIMA, ndiffs, nsdiffs

import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path(r"...")
DATA_FILE = DATA_DIR / "test_set_46.parquet"
OUTPUT_DIR = DATA_DIR / "arima_sarimax_46_rolling_outputsv2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COL = "DATE"
TARGET_COL = "SALES_QTY"

SEASONAL_PERIOD = 7
FORECAST_HORIZON = 7
TEST_DAYS = 182

MAX_P = 3
MAX_Q = 3
MAX_P_SEASONAL = 1
MAX_Q_SEASONAL = 1

MAX_D = 1
MAX_SEASONAL_D = 1

TEST_ONE_WINDOW = False
TEST_ONE_SERIES = False

POTENTIAL_EXOG_COLS = [
    "PROMO_FLAG",
    "HOLIDAY_FLAG",
    "AVG_TEMP",
    "AVG_FEEL_TEMP",
    "TOTAL_PRECIP",
    "AVG_WIND",
    "YESTERDAY_HOLIDAY_FLAG",
    "TOMORROW_HOLIDAY_FLAG",
    "DAY_OF_WEEK",
    "WEEKEND_FLAG",
]


Įkeliami duomenys ir surikiuojamos laiko eilutės pagal seriją bei datą.


In [ ]:
main_selected = pd.read_parquet(DATA_FILE)
main_selected[DATE_COL] = pd.to_datetime(main_selected[DATE_COL])
main_selected = main_selected.sort_values(["unique_id", DATE_COL]).reset_index(drop=True)


Atrenkami reikalingi stulpeliai, sutvarkomi išoriniai kintamieji ir pilnas dienų indeksas.


In [ ]:
META_COLS = [
    "sales_level",
    "zero_level",
    "series_type_2d",
    "mean_sales",
    "zero_pct",
]

ID_COLS = [
    "STORE_ID",
    "SKU_GROUP",
    "SKU_ID",
    "unique_id",
]

NON_EXOG_COLS = ID_COLS + META_COLS + [DATE_COL, TARGET_COL]
FORBIDDEN_EXOG_COLS = set(NON_EXOG_COLS)

EXOG_COLS = [
    col for col in POTENTIAL_EXOG_COLS
    if col in main_selected.columns and col not in FORBIDDEN_EXOG_COLS
]

keep_cols = [
    "unique_id",
    DATE_COL,
    TARGET_COL,
    "STORE_ID",
    "SKU_GROUP",
    "SKU_ID",
]

for col in META_COLS:
    if col in main_selected.columns:
        keep_cols.append(col)

keep_cols += EXOG_COLS
keep_cols = list(dict.fromkeys(keep_cols))

Y_df = main_selected[keep_cols].copy()
Y_df = Y_df.rename(columns={DATE_COL: "ds", TARGET_COL: "y"})
Y_df["ds"] = pd.to_datetime(Y_df["ds"])
Y_df["y"] = pd.to_numeric(Y_df["y"], errors="coerce")

agg_dict = {
    "y": ("y", "sum"),
    "STORE_ID": ("STORE_ID", "first"),
    "SKU_GROUP": ("SKU_GROUP", "first"),
    "SKU_ID": ("SKU_ID", "first"),
}

for col in META_COLS:
    if col in Y_df.columns:
        agg_dict[col] = (col, "first")

for col in EXOG_COLS:
    agg_dict[col] = (col, "first")

Y_df = (
    Y_df
    .groupby(["unique_id", "ds"], as_index=False)
    .agg(**agg_dict)
)

all_dates = pd.date_range(Y_df["ds"].min(), Y_df["ds"].max(), freq="D")
full_index = pd.MultiIndex.from_product(
    [sorted(Y_df["unique_id"].unique()), all_dates],
    names=["unique_id", "ds"],
)

metadata_cols = ["unique_id", "STORE_ID", "SKU_GROUP", "SKU_ID"]
for col in META_COLS:
    if col in Y_df.columns:
        metadata_cols.append(col)

metadata = Y_df[metadata_cols].drop_duplicates("unique_id")

Y_df = (
    Y_df
    .set_index(["unique_id", "ds"])
    .reindex(full_index)
    .reset_index()
)

Y_df = Y_df.merge(metadata, on="unique_id", how="left", suffixes=("", "_meta"))
for col in ["STORE_ID", "SKU_GROUP", "SKU_ID"] + META_COLS:
    meta_col = f"{col}_meta"
    if meta_col in Y_df.columns:
        Y_df[col] = Y_df[col].combine_first(Y_df[meta_col])
        Y_df = Y_df.drop(columns=[meta_col])

if "DAY_OF_WEEK" in Y_df.columns:
    Y_df["DAY_OF_WEEK"] = pd.to_numeric(Y_df["DAY_OF_WEEK"], errors="coerce")
    Y_df["DAY_OF_WEEK"] = Y_df["DAY_OF_WEEK"].fillna(Y_df["ds"].dt.dayofweek)
    Y_df["DAY_OF_WEEK"] = Y_df["DAY_OF_WEEK"].astype(int)

    EXOG_COLS = [col for col in EXOG_COLS if col != "DAY_OF_WEEK"]
    Y_df = pd.get_dummies(Y_df, columns=["DAY_OF_WEEK"], prefix="DOW", drop_first=True)

    dow_cols = [f"DOW_{i}" for i in range(1, 7)]
    for col in dow_cols:
        if col not in Y_df.columns:
            Y_df[col] = 0

    EXOG_COLS = EXOG_COLS + dow_cols
    EXOG_COLS = list(dict.fromkeys(EXOG_COLS))

EXOG_COLS = [col for col in EXOG_COLS if col in Y_df.columns]

BINARY_BASE_COLS = [
    "PROMO_FLAG",
    "HOLIDAY_FLAG",
    "YESTERDAY_HOLIDAY_FLAG",
    "TOMORROW_HOLIDAY_FLAG",
    "WEEKEND_FLAG",
]

binary_cols = [col for col in BINARY_BASE_COLS if col in EXOG_COLS]
binary_cols += [col for col in EXOG_COLS if col.startswith("DOW_")]
binary_cols = list(dict.fromkeys(binary_cols))
continuous_exog_cols = [col for col in EXOG_COLS if col not in binary_cols]

for col in EXOG_COLS:
    Y_df[col] = pd.to_numeric(Y_df[col], errors="coerce")

for col in binary_cols:
    Y_df[col] = Y_df[col].fillna(0).astype(float)

for col in continuous_exog_cols:
    Y_df[col] = Y_df[col].astype(float)


Sudaromi 7 dienų slenkančio testavimo langai.


In [ ]:
max_date = Y_df["ds"].max()
test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)

window_starts = [test_start + pd.Timedelta(days=i * FORECAST_HORIZON) for i in range(TEST_DAYS // FORECAST_HORIZON)]
rolling_windows = []
for window_idx, start in enumerate(window_starts, start=1):
    end = start + pd.Timedelta(days=FORECAST_HORIZON - 1)
    rolling_windows.append({"window": window_idx, "start": start, "end": end})


Apibrėžiamos funkcijos diferencijavimo eilėms parinkti.


In [ ]:
def safe_ndiffs(y, test_name: str, max_d: int = 1) -> int:
    y = pd.Series(y).dropna().astype(float)
    return int(ndiffs(y, test=test_name, max_d=max_d))


def safe_nsdiffs(y, test_name: str, m: int = 7, max_D: int = 1) -> int:
    y = pd.Series(y).dropna().astype(float)
    return int(nsdiffs(y, m=m, test=test_name, max_D=max_D))


def choose_differencing(y_train: pd.Series) -> dict:
    d_adf = safe_ndiffs(y_train, "adf", max_d=MAX_D)
    d_kpss = safe_ndiffs(y_train, "kpss", max_d=MAX_D)
    d_chosen = min(max(d_adf, d_kpss), MAX_D)
    D_ocsb = safe_nsdiffs(y_train, "ocsb", m=SEASONAL_PERIOD, max_D=MAX_SEASONAL_D)

    return {
        "d_adf": d_adf,
        "d_kpss": d_kpss,
        "d_chosen": d_chosen,
        "D_chosen": D_ocsb,
    }


Apibrėžiamos funkcijos išoriniams kintamiesiems ir ARIMA/SARIMAX eilėms parinkti.


In [ ]:
def prepare_exog_matrices(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame | None = None
) -> tuple[pd.DataFrame, pd.DataFrame | None]:

    X_train = train_df[EXOG_COLS].astype(float).copy()
    X_test = test_df[EXOG_COLS].astype(float).copy() if test_df is not None else None

    return X_train, X_test


def fit_auto_arima_model(
    y_train: pd.Series,
    d: int,
    D: int,
    X_train: np.ndarray | None = None
):
    model_kwargs = {}

    if X_train is not None:
        model_kwargs["X"] = X_train

    return auto_arima(
        y_train,
        seasonal=True,
        m=SEASONAL_PERIOD,
        d=d,
        D=D,
        start_p=0,
        start_q=0,
        max_p=MAX_P,
        max_q=MAX_Q,
        start_P=0,
        start_Q=0,
        max_P=MAX_P_SEASONAL,
        max_Q=MAX_Q_SEASONAL,
        stepwise=True,
        trace=False,
        error_action="ignore",
        suppress_warnings=True,
        information_criterion="aic",
        with_intercept="auto",
        **model_kwargs,
    )


def fit_fixed_arima_model(
    y_train: pd.Series,
    order: tuple,
    seasonal_order: tuple,
    with_intercept: bool,
    X_train: np.ndarray | None = None
):
    model = ARIMA(
        order=order,
        seasonal_order=seasonal_order,
        with_intercept=with_intercept,
        suppress_warnings=True,
    )

    fit_kwargs = {}

    if X_train is not None:
        fit_kwargs["X"] = X_train

    return model.fit(y_train, **fit_kwargs)


def get_model_with_intercept(model) -> bool:
    with_intercept = getattr(model, "with_intercept", True)

    if with_intercept == "auto":
        return True

    return bool(with_intercept)


def select_initial_orders_for_series(ts: pd.DataFrame) -> dict:
    ts = ts.sort_values("ds").copy()

    unique_id = ts["unique_id"].iloc[0]
    store_id = ts["STORE_ID"].iloc[0]
    sku_group = ts["SKU_GROUP"].iloc[0]
    sku_id = ts["SKU_ID"].iloc[0]
    series_type = ts["series_type_2d"].iloc[0] if "series_type_2d" in ts.columns else np.nan

    initial_train_df = ts[ts["ds"] < test_start].copy()
    y_initial = initial_train_df.set_index("ds")["y"].astype(float)

    diff_info = choose_differencing(y_initial)
    d = diff_info["d_chosen"]
    D = diff_info["D_chosen"]

    order_info = {
        "unique_id": unique_id,
        "STORE_ID": store_id,
        "SKU_GROUP": sku_group,
        "SKU_ID": sku_id,
        "series_type_2d": series_type,
        "d": d,
        "D": D,
        **diff_info,
        "SARIMA_order": None,
        "SARIMA_seasonal_order": None,
        "SARIMA_with_intercept": np.nan,
        "SARIMA_AIC_initial_train": np.nan,
        "SARIMA_selection_error": None,
        "SARIMAX_order": None,
        "SARIMAX_seasonal_order": None,
        "SARIMAX_with_intercept": np.nan,
        "SARIMAX_AIC_initial_train": np.nan,
        "SARIMAX_exog_cols": ", ".join(EXOG_COLS),
        "SARIMAX_selection_error": None,
    }


    sarima_model = fit_auto_arima_model(
        y_train=y_initial,
        d=d,
        D=D,
        X_train=None,
    )

    order_info["SARIMA_order"] = sarima_model.order
    order_info["SARIMA_seasonal_order"] = sarima_model.seasonal_order
    order_info["SARIMA_with_intercept"] = get_model_with_intercept(sarima_model)
    order_info["SARIMA_AIC_initial_train"] = sarima_model.aic()


    X_initial, _ = prepare_exog_matrices(initial_train_df)

    sarimax_model = fit_auto_arima_model(
        y_train=y_initial,
        d=d,
        D=D,
        X_train=X_initial.to_numpy(),
    )

    order_info["SARIMAX_order"] = sarimax_model.order
    order_info["SARIMAX_seasonal_order"] = sarimax_model.seasonal_order
    order_info["SARIMAX_with_intercept"] = get_model_with_intercept(sarimax_model)
    order_info["SARIMAX_AIC_initial_train"] = sarimax_model.aic()

    return order_info


def selected_order_output_row(order_info: dict) -> dict:
    output_cols = [
        "unique_id",
        "STORE_ID",
        "SKU_GROUP",
        "SKU_ID",
        "series_type_2d",
        "d",
        "D",
        "SARIMA_order",
        "SARIMA_seasonal_order",
        "SARIMA_with_intercept",
        "SARIMA_AIC_initial_train",
        "SARIMA_selection_error",
        "SARIMAX_order",
        "SARIMAX_seasonal_order",
        "SARIMAX_with_intercept",
        "SARIMAX_AIC_initial_train",
        "SARIMAX_exog_cols",
        "SARIMAX_selection_error",
    ]

    row = {col: order_info.get(col, np.nan) for col in output_cols}

    tuple_cols = [
        "SARIMA_order",
        "SARIMA_seasonal_order",
        "SARIMAX_order",
        "SARIMAX_seasonal_order",
    ]

    for col in tuple_cols:
        if row[col] is not None:
            row[col] = str(row[col])

    return row


Apibrėžiamas vienos laiko eilutės slenkantis SARIMA ir SARIMAX vertinimas.


In [ ]:
def evaluate_one_series_rolling(ts: pd.DataFrame, windows: list[dict] | None = None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    ts = ts.sort_values("ds").copy()

    unique_id = ts["unique_id"].iloc[0]
    store_id = ts["STORE_ID"].iloc[0]
    sku_group = ts["SKU_GROUP"].iloc[0]
    sku_id = ts["SKU_ID"].iloc[0]
    series_type = ts["series_type_2d"].iloc[0] if "series_type_2d" in ts.columns else np.nan

    order_info = select_initial_orders_for_series(ts)
    selected_order_df = pd.DataFrame([selected_order_output_row(order_info)])

    diff_info = {
        "d_adf": order_info["d_adf"],
        "d_kpss": order_info["d_kpss"],
        "d_chosen": order_info["d_chosen"],
        "D_chosen": order_info["D_chosen"],
    }

    windows_to_run = rolling_windows if windows is None else windows

    forecasts = []
    window_results = []

    for window in windows_to_run:
        window_idx = window["window"]
        start = window["start"]
        end = window["end"]

        train_df = ts[ts["ds"] < start].copy()
        test_df = ts[(ts["ds"] >= start) & (ts["ds"] <= end)].copy()

        y_train = train_df.set_index("ds")["y"].astype(float)
        y_test = test_df.set_index("ds")["y"].astype(float)

        horizon = len(y_test)


        X_train_df, X_test_df = prepare_exog_matrices(train_df, test_df)
        X_train = X_train_df.to_numpy()
        X_test = X_test_df.to_numpy()

        sarima_start = time.time()
        sarima_model = fit_fixed_arima_model(
            y_train=y_train,
            order=order_info["SARIMA_order"],
            seasonal_order=order_info["SARIMA_seasonal_order"],
            with_intercept=order_info["SARIMA_with_intercept"],
            X_train=None,
        )
        sarima_pred_raw = sarima_model.predict(n_periods=horizon)
        sarima_aic = sarima_model.aic()
        sarima_seconds = time.time() - sarima_start

        sarimax_start = time.time()

        sarimax_model = fit_fixed_arima_model(
            y_train=y_train,
            order=order_info["SARIMAX_order"],
            seasonal_order=order_info["SARIMAX_seasonal_order"],
            with_intercept=order_info["SARIMAX_with_intercept"],
            X_train=X_train,
        )

        sarimax_pred_raw = sarimax_model.predict(
            n_periods=horizon,
            X=X_test,
        )
        sarimax_aic = sarimax_model.aic()
        sarimax_seconds = time.time() - sarimax_start

        window_results.append({
            "unique_id": unique_id,
            "STORE_ID": store_id,
            "SKU_GROUP": sku_group,
            "SKU_ID": sku_id,
            "series_type_2d": series_type,
            "window": window_idx,
            "train_end": start - pd.Timedelta(days=1),
            "test_start": start,
            "test_end": end,
            "n_train": len(y_train),
            "n_test": len(y_test),
            **diff_info,
            "SARIMA_order": str(order_info["SARIMA_order"]),
            "SARIMA_seasonal_order": str(order_info["SARIMA_seasonal_order"]),
            "SARIMA_with_intercept": order_info["SARIMA_with_intercept"],
            "SARIMA_AIC": sarima_aic,
            "SARIMA_AIC_initial_train": order_info["SARIMA_AIC_initial_train"],
            "SARIMA_seconds": sarima_seconds,
            "SARIMAX_order": str(order_info["SARIMAX_order"]),
            "SARIMAX_seasonal_order": str(order_info["SARIMAX_seasonal_order"]),
            "SARIMAX_with_intercept": order_info["SARIMAX_with_intercept"],
            "SARIMAX_AIC": sarimax_aic,
            "SARIMAX_AIC_initial_train": order_info["SARIMAX_AIC_initial_train"],
            "SARIMAX_seconds": sarimax_seconds,
        })

        forecast_window_df = pd.DataFrame({
            "unique_id": unique_id,
            "STORE_ID": store_id,
            "SKU_GROUP": sku_group,
            "SKU_ID": sku_id,
            "series_type_2d": series_type,
            "window": window_idx,
            "ds": y_test.index,
            "actual": y_test.values,
            "sarima_forecast_raw": sarima_pred_raw,
            "sarimax_forecast_raw": sarimax_pred_raw,
        })

        forecasts.append(forecast_window_df)

    window_results_df = pd.DataFrame(window_results)
    forecast_result_df = pd.concat(forecasts, ignore_index=True) if forecasts else pd.DataFrame()

    return window_results_df, forecast_result_df, selected_order_df


Paleidžiamas apmokymas visoms pasirinktoms laiko eilutėms iš testinės aibės.


In [ ]:
all_window_results = []
all_forecasts = []
all_selected_orders = []
failed_series = []

unique_ids_all = sorted(Y_df["unique_id"].unique())
selected_unique_ids = unique_ids_all[:1] if (TEST_ONE_WINDOW or TEST_ONE_SERIES) else unique_ids_all
active_windows = rolling_windows[:1] if TEST_ONE_WINDOW else rolling_windows

expected_windows_per_series = len(active_windows)
expected_total_windows = len(selected_unique_ids) * expected_windows_per_series

print("Selected unique_id count:", len(selected_unique_ids))
print("Expected windows per series:", expected_windows_per_series)
print("Expected total windows:", expected_total_windows)
print("Full dataset unique_id count:", len(unique_ids_all))
print("Full-run expected total windows:", len(unique_ids_all) * (TEST_DAYS // FORECAST_HORIZON))

for i, unique_id in enumerate(selected_unique_ids, start=1):
    print(f"[{i}/{len(selected_unique_ids)}] Processing {unique_id}")
    ts = Y_df[Y_df["unique_id"] == unique_id].copy()

    try:
        result_df, forecast_df, selected_order_df = evaluate_one_series_rolling(ts, windows=active_windows)
        all_window_results.append(result_df)
        all_forecasts.append(forecast_df)
        all_selected_orders.append(selected_order_df)
    except Exception as exc:
        print(f"FAILED: {unique_id}: {exc}")
        failed_series.append({"unique_id": unique_id, "error_message": str(exc)})


Sujungiami rezultatai ir išvedama trumpa modelių parinkimo santrauka.


In [ ]:
window_results_df = pd.concat(all_window_results, ignore_index=True) if all_window_results else pd.DataFrame()
forecasts_df = pd.concat(all_forecasts, ignore_index=True) if all_forecasts else pd.DataFrame()
selected_orders_df = pd.concat(all_selected_orders, ignore_index=True) if all_selected_orders else pd.DataFrame()
failed_df = pd.DataFrame(failed_series)

print("Successful series:", window_results_df["unique_id"].nunique() if len(window_results_df) else 0)
print("Failed series:", len(failed_df))

if len(selected_orders_df) > 0:
    print("\nSelected order summary:")
    print(
        selected_orders_df
        .groupby(["SARIMA_order", "SARIMA_seasonal_order"], dropna=False)
        .size()
        .reset_index(name="n_series")
        .sort_values("n_series", ascending=False)
        .head(20)
    )

    print("\nSelected SARIMAX order summary:")
    print(
        selected_orders_df
        .groupby(["SARIMAX_order", "SARIMAX_seasonal_order"], dropna=False)
        .size()
        .reset_index(name="n_series")
        .sort_values("n_series", ascending=False)
        .head(20)
    )

if len(selected_orders_df) > 0:
    print("Chosen non-seasonal differencing d:")
    print(selected_orders_df["d"].value_counts(dropna=False).sort_index())

    print("\nChosen seasonal differencing D:")
    print(selected_orders_df["D"].value_counts(dropna=False).sort_index())


Išsaugomos SARIMA ir SARIMAX prognozės į parquet failą.


In [ ]:
forecasts_path = OUTPUT_DIR / "sarima_sarimax_46_rolling_forecasts.parquet"

if len(forecasts_df) > 0:
    forecasts_df.to_parquet(forecasts_path, index=False)
